In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, TimestampType
from pyspark.sql.functions import current_timestamp, to_timestamp, concat, col, lit

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_au")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlsgr18smartdata")

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/transactions.csv"

In [0]:
# Matches transactions.csv header order: date,store_nbr,transactions
transactions_schema = StructType(fields=[StructField("date", TimestampType(), True),
                                  StructField("store_nbr", IntegerType(), True),
                                  StructField("transactions", IntegerType(), True)
])

In [0]:
transactions_df = spark.read \
            .option("header", True) \
            .schema(transactions_schema) \
            .csv(ruta)

In [0]:
# Target table has no ingestion_date column, so no extra columns are added
transactions_with_timestamp_df = transactions_df

In [0]:
# Column order must match catalog_au.bronze.transactions (insertInto is positional)
transactions_selected_df = transactions_with_timestamp_df.select(col('date'),
                                                   col('store_nbr'),
                                                   col('transactions'))

In [0]:
# insertInto keeps the table definition from 1.Preparacion_Ambiente
transactions_selected_df.write.mode('overwrite').insertInto(f'{catalogo}.{esquema}.transactions')